[Start](../README.md) · [Demos](README.md) · [API-Zugang](../docs/API_ZUGANG.md)

# Demo 2 · REST-APIs: CoinGecko und Flickr

**Lernziele:** Du kannst Endpunkt, Parameter, API-Schlüssel und Antwortformat auseinanderhalten, JSON in Tabellen überführen und einen HTTP-Erfolg von einem fachlichen API-Erfolg unterscheiden.

Wir betrachten zwei Datenformen: **CoinGecko** liefert Werte unter Coin-IDs; **Flickr** liefert eine paginierte Liste von Fotometadaten. Die öffentliche Flickr-Suche benötigt einen Anwendungsschlüssel, aber für diesen Aufruf weder Benutzer-OAuth noch ein API-Secret.

**Standard: synthetische Beispieldaten.** Sämtliche Preise, IDs, Titel und Besitzerangaben der lokalen Beispiele sind erfunden. Es werden keine Fotos heruntergeladen. Du kannst die Demo ohne Konto, Schlüssel oder Browsercontainer ausführen. Live-Aufrufe aktivierst Du für jeden Anbieter separat; jeder aktive Abschnitt sendet höchstens eine Anfrage pro Ausführung.

## 1 · Vorbereitung und unabhängige Schalter

Wähle **Python (webscraping-workshop)**. Für Live-Aufrufe ergänze Deine vorhandene `.env` gemäss [API-Zugang](../docs/API_ZUGANG.md). Das Notebook lädt die Datei mit `override=False`: Bereits gesetzte Umgebungsvariablen behalten Vorrang.

Gib keine Schlüssel, vollständigen Flickr-Anfrage-URLs oder Request-Objekte aus. Flickr übermittelt seinen Anwendungsschlüssel als Anfrageparameter; deshalb unterdrücken wir URL-haltige Exceptiontexte. Die Exporte enthalten nur ausdrücklich ausgewählte Herkunftsfelder.

In [1]:
from pathlib import Path
import sys

# Funktioniert im Repo-Hauptordner und in dessen Unterordnern.
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Öffne das Notebook innerhalb des Workshop-Repos.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repo gefunden:", ROOT.name)

Repo gefunden: webscraping-workshop-2026


In [2]:
import json
import math
import os
from datetime import datetime, timezone
from numbers import Real

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

load_dotenv(ROOT / ".env", override=False)
COINGECKO_LIVE = False
FLICKR_LIVE = False

COINGECKO_URL = "https://api.coingecko.com/api/v3/simple/price"
FLICKR_URL = "https://www.flickr.com/services/rest/"
COIN_IDS = ("bitcoin", "ethereum")  # Eindeutige Anbieter-IDs, keine Börsenticker.
QUOTE_CURRENCIES = ("eur", "chf")
FLICKR_SEARCH = "Winterthur"
FLICKR_PER_PAGE = 5

if not COIN_IDS or len(COIN_IDS) > 10 or len(COIN_IDS) != len(set(COIN_IDS)):
    raise ValueError("Verwende 1 bis 10 verschiedene Coin-IDs.")
if not QUOTE_CURRENCIES or len(QUOTE_CURRENCIES) != len(set(QUOTE_CURRENCIES)):
    raise ValueError("Die Zielwährungen müssen eindeutig und nicht leer sein.")
if not 1 <= FLICKR_PER_PAGE <= 5:
    raise ValueError("Die Demo ist auf höchstens 5 Fotos einer Seite begrenzt.")
print("CoinGecko:", "LIVE" if COINGECKO_LIVE else "SYNTHETISCHES SAMPLE")
print("Flickr:", "LIVE" if FLICKR_LIVE else "SYNTHETISCHES SAMPLE")

CoinGecko: SYNTHETISCHES SAMPLE
Flickr: SYNTHETISCHES SAMPLE


## 2 · Gemeinsame HTTP-Regeln sichtbar machen

Der Helfer kapselt nur den wiederkehrenden HTTP-Abruf. Die anbieterbezogenen Parameter und die JSON-Verarbeitung bleiben in den folgenden Zellen sichtbar.

- Verbindung und Antwort erhalten begrenzte Wartezeiten.
- HTTP 401/403, 429 und Serverfehler führen zu einer verständlichen Meldung.
- Weiterleitungen werden nicht automatisch verfolgt.
- Fehler brechen ab; es gibt **keinen stillen Rückfall** zu Beispieldaten.
- Wir geben weder Antworttext noch ursprüngliche Exceptiontexte aus, weil sie Anfrageinformationen enthalten könnten.

Bei 429 wartest Du und prüfst Dein Anbieter-Kontingent, statt die Zelle laufend erneut zu starten.

In [3]:
def fetch_json(endpoint, *, provider, params, headers=None):
    try:
        request_headers = {"User-Agent": "WebscrapingWorkshop/0.2 (educational exercise)", **(headers or {})}
        with requests.get(endpoint, params=params, headers=request_headers,
                          timeout=(5, 20), allow_redirects=False) as response:
            status = response.status_code
            if status != 200:
                if status in (401, 403):
                    hint = "Prüfe API-Schlüssel, Berechtigung und Tarif."
                elif status == 429:
                    hint = "Abruflimit erreicht; warte und prüfe Dein Kontingent."
                elif 500 <= status <= 599:
                    hint = "Anbieter vorübergehend nicht verfügbar."
                else:
                    hint = "Prüfe Endpunkt und Parameter in der Dokumentation."
                raise RuntimeError(f"{provider}: HTTP {status}. {hint}")
            try:
                payload = response.json()
            except ValueError:
                raise ValueError(f"{provider}: Antwort ist kein gültiges JSON.") from None
    except requests.RequestException:
        raise RuntimeError(f"{provider}: Netzwerkfehler oder Zeitüberschreitung; keine Daten geladen.") from None
    if not isinstance(payload, dict):
        raise ValueError(f"{provider}: Ein JSON-Objekt wurde erwartet.")
    return payload


def read_synthetic_sample(filename):
    document = json.loads((ROOT / "data" / "samples" / filename).read_text(encoding="utf-8"))
    if document.get("_sample_metadata", {}).get("synthetic") is not True:
        raise ValueError("Die Beispieldatei ist nicht als synthetisch gekennzeichnet.")
    # Die äussere Hülle dokumentiert die Herkunft; payload bildet die API-Antwort nach.
    return document["payload"]

## 3 · CoinGecko: Parameter und Datenbezug

`ids` wählt Anbieter-IDs; `vs_currencies` bestimmt die Zielwährungen. Mit `include_last_updated_at` fordern wir zusätzlich den Aktualisierungszeitpunkt an. Für den Demo-Zugang gehört der Schlüssel in den Header **`x-cg-demo-api-key`**.

Im Sample-Modus sind `bitcoin` und `ethereum` in EUR und CHF enthalten. Für andere Parameter brauchst Du passende synthetische Daten oder einen expliziten Live-Abruf.

In [4]:
coingecko_mode = "live" if COINGECKO_LIVE else "sample"
coingecko_params = {
    "ids": ",".join(COIN_IDS),
    "vs_currencies": ",".join(QUOTE_CURRENCIES),
    "include_last_updated_at": "true",
}
if COINGECKO_LIVE:
    if not os.getenv("COINGECKO_DEMO_API_KEY", "").strip():
        raise RuntimeError("Live-Modus: COINGECKO_DEMO_API_KEY fehlt in .env bzw. der Umgebung.")
    coingecko_payload = fetch_json(
        COINGECKO_URL, provider="CoinGecko", params=coingecko_params,
        headers={"x-cg-demo-api-key": os.environ["COINGECKO_DEMO_API_KEY"].strip()},
    )
else:
    coingecko_payload = read_synthetic_sample("coingecko_sample.json")
coingecko_acquired_at = datetime.now(timezone.utc).isoformat()
print("CoinGecko-Antwort geladen; Modus:", coingecko_mode)

CoinGecko-Antwort geladen; Modus: sample


## 4 · CoinGecko: Verschachtelung in Zeilen überführen

Die fachliche Zeile ist eine Kombination aus **Coin-ID und Zielwährung**. Zwei Coins in zwei Währungen ergeben vier Zeilen. Ein ausgelassener Coin oder ein `null`-Preis wird nicht als null Franken bzw. Euro interpretiert.

`last_updated_at` ist eine UNIX-Zeit in Sekunden. Wir wandeln sie explizit in UTC um. Die Aktualisierung beim Anbieter und unser eigener Datenbezug bleiben zwei getrennte Felder.

In [5]:
def parse_coingecko(payload, coin_ids, currencies):
    if not isinstance(payload, dict):
        raise ValueError("CoinGecko: Ein JSON-Objekt wurde erwartet.")
    if "error" in payload or "status" in payload:
        raise ValueError("CoinGecko meldet einen API-Fehler; prüfe Schlüssel, Tarif und Parameter.")
    rows = []
    for coin_id in coin_ids:
        values = payload.get(coin_id)
        if not isinstance(values, dict):
            raise ValueError(f"CoinGecko: Erwartete Coin-ID {coin_id} fehlt.")
        updated = values.get("last_updated_at")
        if isinstance(updated, bool) or not isinstance(updated, Real) or not math.isfinite(updated) or updated <= 0:
            raise ValueError("CoinGecko: Gültiger Aktualisierungszeitpunkt fehlt.")
        try:
            updated_iso = datetime.fromtimestamp(updated, tz=timezone.utc).isoformat()
        except (OverflowError, OSError, ValueError):
            raise ValueError("CoinGecko: Aktualisierungszeitpunkt ausserhalb des gültigen Bereichs.") from None
        for currency in currencies:
            price = values.get(currency)
            if isinstance(price, bool) or not isinstance(price, Real) or not math.isfinite(price) or price <= 0:
                raise ValueError(f"CoinGecko: Gültiger Preis für {coin_id}/{currency} fehlt.")
            rows.append({"coin_id": coin_id, "quote_currency": currency.upper(),
                         "price": float(price), "provider_updated_at_utc": updated_iso})
    result = pd.DataFrame(rows)
    if result.empty or result.duplicated(["coin_id", "quote_currency"]).any():
        raise ValueError("CoinGecko: Leere oder doppelte Coin-Währungs-Kombinationen.")
    return result

coin_prices = parse_coingecko(coingecko_payload, COIN_IDS, QUOTE_CURRENCIES)
coin_prices["source_mode"] = coingecko_mode
coin_prices["acquired_at_utc"] = coingecko_acquired_at
display(coin_prices)

,coin_id,quote_currency,price,provider_updated_at_utc,source_mode,acquired_at_utc
0,bitcoin,EUR,50000.0,2026-09-01T08:00:00+00:00,sample,2026-09-30T10:59:10.436793+00:00
1,bitcoin,CHF,48000.0,2026-09-01T08:00:00+00:00,sample,2026-09-30T10:59:10.436793+00:00
2,ethereum,EUR,2500.0,2026-09-01T08:00:00+00:00,sample,2026-09-30T10:59:10.436793+00:00
3,ethereum,CHF,2400.0,2026-09-01T08:00:00+00:00,sample,2026-09-30T10:59:10.436793+00:00


## 5 · Flickr: eine begrenzte Seite mit Metadaten

Die Methode `flickr.photos.search` liefert öffentliche Fotos zu unserem Suchtext. Wir fordern **Seite 1 mit höchstens fünf Treffern** und zusätzlich die Lizenz-ID an. `format=json` wählt JSON; `nojsoncallback=1` vermeidet eine umschliessende JavaScript-Funktion (JSONP).

Ein Anwendungsschlüssel identifiziert hier die aufrufende Anwendung. Benutzer-OAuth wäre ein anderes Verfahren, beispielsweise für den Zugriff auf berechtigte nichtöffentliche Inhalte. **Ein sichtbares Foto ist nicht automatisch frei verwendbar:** Lizenz-ID, Lizenztext, Urheberschaft und weitere Bedingungen müssen vor einer Nutzung geprüft werden.

Das Sample enthält fest drei erfundene Treffer zum Lehrbeispiel; geänderte Suchparameter werden darauf nicht angewendet. Passe für andere Übungssuchen die Beispieldatei an oder aktiviere den Live-Modus.

In [6]:
flickr_mode = "live" if FLICKR_LIVE else "sample"
# Dieses Dictionary enthält bewusst keinen Schlüssel und darf in die Metadaten.
flickr_public_params = {
    "method": "flickr.photos.search", "text": FLICKR_SEARCH,
    "media": "photos", "safe_search": 1,
    "page": 1, "per_page": FLICKR_PER_PAGE,
    "extras": "license", "format": "json", "nojsoncallback": 1,
}
if FLICKR_LIVE:
    if not os.getenv("FLICKR_API_KEY", "").strip():
        raise RuntimeError("Live-Modus: FLICKR_API_KEY fehlt in .env bzw. der Umgebung.")
    flickr_payload = fetch_json(
        FLICKR_URL, provider="Flickr",
        params={**flickr_public_params, "api_key": os.environ["FLICKR_API_KEY"].strip()},
    )
else:
    flickr_payload = read_synthetic_sample("flickr_sample.json")
flickr_acquired_at = datetime.now(timezone.utc).isoformat()
print("Flickr-Antwort geladen; Modus:", flickr_mode)

Flickr-Antwort geladen; Modus: sample


## 6 · Flickr: HTTP-Status und API-Status unterscheiden

Flickr kann in einer technisch zugestellten JSON-Antwort einen API-Fehler mit `stat="fail"` melden. Deshalb prüfen wir **zusätzlich zu HTTP 200** das Feld `stat`.

Aus `photos.photo` lesen wir die Datensätze. `total` beschreibt die Trefferzahl der Suche, nicht die Anzahl unserer geladenen Zeilen. Wir laden absichtlich keine Folgeseiten und keine Bilddateien.

In [7]:
def parse_flickr(payload, max_rows=5):
    if not isinstance(payload, dict) or payload.get("stat") != "ok":
        raise ValueError("Flickr: API-Status ist nicht ok. Prüfe Schlüssel, Parameter und Verfügbarkeit.")
    photos = payload.get("photos")
    if not isinstance(photos, dict) or not isinstance(photos.get("photo"), list):
        raise ValueError("Flickr: Erwartete Struktur photos.photo fehlt.")
    try:
        page = int(photos["page"])
        total = int(photos["total"])
    except (KeyError, TypeError, ValueError):
        raise ValueError("Flickr: Seitenzahl oder Gesamttrefferzahl fehlt bzw. ist ungültig.") from None
    items = photos["photo"]
    if page != 1 or total < len(items) or len(items) > max_rows:
        raise ValueError("Flickr: Unerwartete Seite oder widersprüchliche Treffermengen.")
    rows = []
    for item in items:
        if not isinstance(item, dict):
            raise ValueError("Flickr: Fotoeintrag ist kein Objekt.")
        if not all(isinstance(item.get(k), str) and item[k] for k in ("id", "owner")):
            raise ValueError("Flickr: Foto-ID oder Besitzer-ID fehlt.")
        if not isinstance(item.get("title"), str):
            raise ValueError("Flickr: Titel fehlt oder hat einen unerwarteten Datentyp.")
        license_id = item.get("license")
        if not isinstance(license_id, (str, int)) or isinstance(license_id, bool) or not str(license_id).isdigit():
            raise ValueError("Flickr: Angeforderte Lizenz-ID fehlt oder ist ungültig.")
        rows.append({"photo_id": item["id"], "owner_id": item["owner"],
                     "title": item["title"], "license_id": str(license_id)})
    result = pd.DataFrame(rows, columns=["photo_id", "owner_id", "title", "license_id"])
    if result["photo_id"].duplicated().any():
        raise ValueError("Flickr: Doppelte Foto-ID innerhalb der geladenen Seite.")
    return result, total

photos, total_matches = parse_flickr(flickr_payload, FLICKR_PER_PAGE)
photos["source_mode"] = flickr_mode
photos["acquired_at_utc"] = flickr_acquired_at
print(f"Geladen: {len(photos)} Datensätze; Gesamttreffer laut Antwort: {total_matches}.")
if photos.empty:
    print("Die Suche liefert keine Treffer. Prüfe Suchbegriff und Antwort, bevor Du weiterarbeitest.")
display(photos)

Geladen: 3 Datensätze; Gesamttreffer laut Antwort: 3.


,photo_id,owner_id,title,license_id,source_mode,acquired_at_utc
0,sample-001,sample-owner-001,Synthetisches Beispiel: Bahnhof Winterthur,0,sample,2026-09-30T10:59:12.536993+00:00
1,sample-002,sample-owner-002,Synthetisches Beispiel: Stadtbibliothek,0,sample,2026-09-30T10:59:12.536993+00:00
2,sample-003,sample-owner-001,Synthetisches Beispiel: Campus,0,sample,2026-09-30T10:59:12.536993+00:00


## 7 · Ergebnisse mit Herkunft exportieren

Jeder Anbieter erhält eine eigene Ablage, getrennt nach `sample` und `live`. Die Metadaten übernehmen nur die **schlüsselfreien** Parameter. Die Rohdaten enthalten den Antwortinhalt; API-Schlüssel, Header und vollständige Anfrage-URLs werden nicht gespeichert.

Die Lizenz-ID bleibt ein Merkmal, keine pauschale Nutzungserlaubnis. Die synthetischen Flickr-Datensätze verweisen auf keine realen Fotos.

In [8]:
exports = [
    ("coingecko", coingecko_mode, coingecko_payload, coin_prices, coingecko_acquired_at,
     COINGECKO_URL if COINGECKO_LIVE else "data/samples/coingecko_sample.json", coingecko_params),
    ("flickr", flickr_mode, flickr_payload, photos, flickr_acquired_at,
     FLICKR_URL if FLICKR_LIVE else "data/samples/flickr_sample.json", flickr_public_params),
]
for provider, mode, payload, frame, acquired_at, source_location, public_params in exports:
    output_dir = ROOT / "data" / "output" / "demo02_apis" / provider / mode
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "raw.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    frame.to_csv(output_dir / "records.csv", index=False, encoding="utf-8")
    metadata = {
        "provider": provider, "source_mode": mode, "synthetic": mode == "sample",
        "source_location": source_location, "acquired_at_utc": acquired_at,
        "request_parameters_without_credentials": public_params,
        "request_parameters_applied_to_live_request": mode == "live",
        "row_count": len(frame),
    }
    (output_dir / "metadata.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    restored = pd.read_csv(output_dir / "records.csv")
    if len(restored) != len(frame):
        raise RuntimeError("Exportprüfung fehlgeschlagen.")
    print("Export geprüft:", output_dir.relative_to(ROOT))

Export geprüft: data\output\demo02_apis\coingecko\sample
Export geprüft: data\output\demo02_apis\flickr\sample


## Reflexion und Transfer

1. Was unterscheiden Coin-ID, Zielwährung und API-Schlüssel? Welche gehören in veröffentlichte Projektdateien?
2. Weshalb kann eine API-Anfrage trotz HTTP 200 fachlich fehlgeschlagen sein?
3. Warum ist `total_matches` nicht dasselbe wie die Zahl unserer exportierten Fotos?
4. Welche Information fehlt, wenn Du nur Preise ohne Anbieterzeitpunkt und Bezugzeitpunkt speicherst?
5. Wie würde sich der Ablauf ändern, wenn Du berechtigt auf nichtöffentliche Flickr-Inhalte zugreifen müsstest?
6. Welche zusätzlichen Entscheidungen bräuchte eine regelmässige Ingestion: Abrufintervall, Kontingent, Pagination, Speicherung und Änderungen des Schemas?

**Kontrollpunkt im Sample-Modus:** vier Coin-Währungs-Zeilen und drei synthetische Fotometadaten-Zeilen. Beide Exporte liegen unter `data/output/demo02_apis/` im jeweiligen Unterordner `sample/`.

**Live separat ausprobieren:** Setze nur den gewünschten Anbieter-Schalter auf `True`, führe die Konfigurationszelle erneut aus und danach dessen Bezug- und Parsingzellen. Für konsistente Exporte danach auch Abschnitt 7 ausführen. Ein vollständiges «Run All» sendet pro aktivem Anbieter erneut eine Anfrage.

## Quellen und Herkunft

Offizielle Dokumentation, geprüft am **30.09.2026**:

- [CoinGecko Demo API: Simple Price](https://docs.coingecko.com/demo/reference/simple-price)
- [CoinGecko Demo API: Authentifizierung](https://docs.coingecko.com/demo/reference/authentication)
- [Flickr: flickr.photos.search](https://www.flickr.com/services/api/flickr.photos.search.html)
- [Flickr: JSON-Antwortformat](https://www.flickr.com/services/api/response.json.html)
- [Flickr: Lizenzinformationen](https://www.flickr.com/services/api/flickr.photos.licenses.getInfo.html)

Die Anwendungsfälle knüpfen an die bereitgestellten Vorjahresnotebooks `goingecko_example.ipynb` und `flickr_example_01.ipynb` an. Diese nennen Gellrich, M. (2021), *Exercises in the theme block “Data Ingestion”*, Spring semester 2021, ZHAW Winterthur, als Grundlage. Code, Schutz vor Schlüsselausgaben und synthetische Daten wurden hier neu ausgearbeitet; vorhandene Zugangsdaten wurden nicht übernommen.

Die Standardausführung prüft lokale Beispieldaten. **Ein erfolgreicher Sample-Lauf belegt keinen erfolgreichen Live-Zugang.**

[Zurück zu den Demos](README.md) · [Weiter: HTML mit BeautifulSoup](03_html_beautifulsoup.ipynb)